# SpotV2Net: Multivariate Intraday Spot Volatility Forecasting

**ArXivist-generated reproduction notebook**

Paper: [arXiv:2401.06249](https://arxiv.org/abs/2401.06249) — Brini & Toscano
Generated: 2026-07-07

This notebook walks through the key components of the implementation, runs a
small-scale training loop on synthetic data, and verifies that the setup matches
the paper's reported behavior on a mini-dataset.


In [ ]:
# Check Python version, GPU availability, and key dependencies
import sys
try:
    import torch
    print(f"Python: {sys.version}")
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("Running on CPU — training will be slow")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError as e:
    print(f"Missing dependency: {e}. Run: pip install -r ../requirements.txt")


In [ ]:
# Install the project in editable mode (run once)
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])


## Paper Overview

**Problem:** forecast the 30-minute-ahead intraday spot volatility of a
cross-section of assets (the 30 DJIA constituents), jointly, rather than asset by
asset.

**Core idea:** represent assets as nodes of a fully-connected graph. Node
features are Fourier non-parametric estimates of each asset's own spot
volatility and its spot co-volatilities with every other asset. Edge features —
the key novelty — are Fourier estimates of spot **volatility-of-volatility** and
**co-volatility-of-volatility**, which the paper argues are natural indicators of
instantaneous spillover strength between two assets' volatilities. A Graph
Attention Network (Velickovic et al., 2017), extended to condition its attention
scores on these edge features, produces the forecast.

**Mapping to this repo:**
- `src/spotv2net/data/fourier_estimators.py` → Sec. 6 (Fourier estimators)
- `src/spotv2net/data/graph_builder.py` → Eq. 1-2 (node/edge feature construction)
- `src/spotv2net/models/gat_layer.py`, `spotv2net.py` → Sec. 5 (GAT architecture)
- `src/spotv2net/training/`, `evaluation/` → Sec. 7.2-7.4 (training & evaluation)


## Component walkthrough: Edge-aware GAT layer

$$\alpha'_{ij} = \frac{\exp\left(\mathrm{LeakyReLU}(q'^T[Wx_i \| Wx_j \| Ux^e_{ij}])\right)}{\sum_{k=1}^{N}\exp\left(\mathrm{LeakyReLU}(q'^T[Wx_i \| Wx_k \| Ux^e_{ik}])\right)}$$

This is the core novelty of the paper (Sec. 5): standard GAT attention extended
to also condition on edge features $x^e_{ij}$ (the vol-of-vol pair).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

import torch
from spotv2net.models.gat_layer import EdgeAwareGATLayer

torch.manual_seed(42)
N, in_dim, edge_dim, out_dim, heads = 6, 16, 9, 8, 4

layer = EdgeAwareGATLayer(in_dim=in_dim, out_dim=out_dim, edge_dim=edge_dim, heads=heads, concat=True)

x = torch.randn(N, in_dim)
# fully connected edges, no self-loops
src = [i for i in range(N) for j in range(N) if i != j]
dst = [j for i in range(N) for j in range(N) if i != j]
edge_index = torch.tensor([src, dst])
edge_attr = torch.randn(edge_index.size(1), edge_dim)

out = layer(x, edge_index, edge_attr)
print(f"Input node features:  {x.shape}")
print(f"Edge features:        {edge_attr.shape}")
print(f"Output node features: {out.shape}  (expected: [{N}, {out_dim}])")


## Component walkthrough: Full SpotV2Net model

Stacks two `EdgeAwareGATLayer`s (Table 8's tuned architecture) and a final linear
prediction head, $\hat{y}_i = Ox'_i + u$.


In [ ]:
from spotv2net.models.spotv2net import SpotV2Net

num_assets = 6          # toy graph size (paper uses 30)
num_lags = 3             # toy lag window (paper-optimal: 42)
node_in_dim = (num_lags + 1) * num_assets
edge_in_dim = 3 * (num_lags + 1)

model = SpotV2Net(
    node_in_dim=node_in_dim,
    edge_in_dim=edge_in_dim,
    hidden_dims=[32, 16],
    heads=4,
    output_dim=1,
    use_edge_features=True,
)
print(model)

x = torch.randn(num_assets, node_in_dim)
src = [i for i in range(num_assets) for j in range(num_assets) if i != j]
dst = [j for i in range(num_assets) for j in range(num_assets) if i != j]
edge_index = torch.tensor([src, dst])
edge_attr = torch.randn(edge_index.size(1), edge_in_dim)

pred = model(x, edge_index, edge_attr)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {pred.shape}")
print(f"Expected:     ({num_assets}, 1)")


## Mini-training demonstration

Trains SpotV2Net for a few steps on synthetic data (no downloads required) to
verify the loss decreases and the training loop runs end to end.


In [ ]:
# Data cell: tiny synthetic dataset (100 samples max, in-memory only)
import numpy as np

n_samples = 40
torch.manual_seed(0)
xs = [torch.randn(num_assets, node_in_dim) for _ in range(n_samples)]
edge_attrs = [torch.randn(edge_index.size(1), edge_in_dim) for _ in range(n_samples)]
ys = [torch.rand(num_assets, 1) * 1e-4 for _ in range(n_samples)]  # volatility-scale targets
print(f"Synthetic dataset: {n_samples} graph snapshots, {num_assets} nodes each")


In [ ]:
# Model init cell
mini_model = SpotV2Net(
    node_in_dim=node_in_dim, edge_in_dim=edge_in_dim,
    hidden_dims=[32, 16], heads=4, output_dim=1, use_edge_features=True,
)
n_params = sum(p.numel() for p in mini_model.parameters())
print(f"Mini-model parameter count: {n_params:,}")

optimizer = torch.optim.AdamW(mini_model.parameters(), lr=1e-3)


In [ ]:
# Training loop cell: run 8 steps, print loss at each step
from spotv2net.training.losses import mse_loss

mini_model.train()
for step in range(8):
    idx = step % n_samples
    optimizer.zero_grad()
    pred = mini_model(xs[idx], edge_index, edge_attrs[idx])
    loss = mse_loss(pred, ys[idx])
    loss.backward()
    optimizer.step()
    print(f"step {step}: loss={loss.item():.6e}")


### Results

The loss above should generally trend downward across the 8 steps (on this tiny
synthetic, noisy dataset it may not be perfectly monotonic — that's expected).
Output shapes should consistently match `(num_assets, 1)`.


## Paper's reported results (from the SIR)


In [ ]:
# Results reported in the paper (SIR evaluation_protocol.reported_results, Table 2, test period)
paper_results = {
    "dataset": "DJIA-30, TAQ 1-second prices, test period 2022-10-15 to 2023-05-10",
    "metric": "MSE / QLIKE (single-step, 30-min-ahead)",
    "SpotV2Net": {"MSE": 4.273e-08, "QLIKE": 0.286},
    "baseline_HAR-Spot": {"MSE": 5.089e-08, "QLIKE": 0.343},
    "baseline_XGB": {"MSE": 6.232e-08, "QLIKE": 0.429},
    "baseline_LSTM": {"MSE": 5.198e-08, "QLIKE": 0.359},
}
print("Paper's claimed results (test period, single-step):")
for k, v in paper_results.items():
    print(f"  {k}: {v}")
print("\nTo reproduce these results, run train.py with the full config and real DJIA data.")
print("Then use the Results Comparator (Stage 6) to compare your outputs.")


## What to do next

1. **Get real data**: see `../data/README_data.md` (WRDS TAQ access required; a
   synthetic smoke-test generator is available via `python data/download.py --synthetic`).
2. **Full training**: `python train.py --config configs/config.yaml`
3. **Evaluation**: `python evaluate.py --config configs/config.yaml --checkpoint checkpoints/best.pt`
4. **Compare results**: feed your results back to ArXivist's Results Comparator (Stage 6).

**Implementation notes from the SIR (top assumptions to be aware of):**
- Fourier cutting frequencies are placeholders — calibrate against the FMVol
  library or Mancino & Recchioni (2015) formulas (confidence 0.35-0.4).
- AdamW betas / weight decay / LR schedule are PyTorch defaults, not disclosed
  in the paper (confidence 0.4-0.55).
- Feature standardization (StandardScaler) is an assumed preprocessing step,
  not explicitly described in the paper (confidence 0.45).
